This notebook is to analyze the BBQ metrics on the original dataset and paraphrased one.

In [1]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tikzplotlib
import sys
from pathlib import Path
notebook_dir = Path().resolve()  
src_path = notebook_dir.parent 
sys.path.append(str(src_path))
from src.configuration import *
from matplotlib.colors import ListedColormap, TwoSlopeNorm

# Data importation

In [62]:
#Define here which dataset you want to analyze
dataset='BBQ'

In [63]:
subsets_dataset=subsets[dataset]
metrics_dataset=metrics[dataset]
OUTPUT_FILE=f"../result/{dataset}/all_results.csv"

In [64]:
result_df=pd.read_csv(OUTPUT_FILE)

In [5]:
#Optional: to analyze results per generation model
#gen_model='chatgpt'
#result_df=result_df[(result_df.generation_model==gen_model)|(result_df.modification=='Original')]

In [65]:
result_df.modification=result_df['modification'].replace(
    {'Random': 'Baseline'}
)

In [66]:
MODEL_ORDER=['MPT-7B', 'MPT-7B-Inst','Falcon-7B', 'Falcon-7B-Inst','Llama-3-8B', 'Llama3-8B-Inst','Gemma3-1B', 'Gemma3-4B', 'Gemma3-12B']
MODIF_ORDER=["Baseline", "Prepositions", "Synonyms", "Voice Change", "Formal Style", "AAE Dialect"]

# Analysis

In [68]:
#Scores results per original, baseline and AUGMENT settings
grouped = result_df[result_df["generation_model"].isna() | (result_df["generation_model"] == "chatgpt")].copy()

# Collapse modification values
grouped["modification"] = grouped["modification"].apply(
    lambda x: x if x in ["Original", "Baseline"] else "All Paraphrases"
)

# Group by target_model + modification
grouped = (
    grouped
    .groupby(["target_model", "modification"], as_index=False)
    .agg({metric: "mean" for metric in metrics_dataset.keys()})
)

for metric in metrics_dataset:
    # Pivot to wide format (one metric at a time)
    print('\n')
    print(metric)
    table=grouped.pivot(index="target_model", columns="modification", values=metric)
    # Reorder columns: Original → All Paraphrases → Random
    table = table[["Original", "All Paraphrases", "Baseline"]]
    table = table.reindex(MODEL_ORDER)

    # Convert to percentages with 2 decimals
    display((table * 100).round(2))



overall_acc


modification,Original,All Paraphrases,Baseline
target_model,,,
MPT-7B,32.11,31.98,32.21
MPT-7B-Inst,31.68,31.99,32.27
Falcon-7B,28.75,28.90,28.82
Falcon-7B-Inst,29.61,29.56,29.60
Llama-3-8B,40.96,40.95,40.59
Llama3-8B-Inst,33.56,33.68,33.82
Gemma3-1B,32.12,32.09,32.02
Gemma3-4B,57.62,57.37,57.29
Gemma3-12B,82.45,81.99,81.37




ambig_acc


modification,Original,All Paraphrases,Baseline
target_model,,,
MPT-7B,23.15,23.16,23.60
MPT-7B-Inst,22.15,22.35,22.45
Falcon-7B,15.41,15.86,15.64
Falcon-7B-Inst,19.25,19.39,19.07
Llama-3-8B,24.72,25.03,25.31
Llama3-8B-Inst,36.76,37.77,37.57
Gemma3-1B,30.10,29.55,29.57
Gemma3-4B,46.27,45.13,45.49
Gemma3-12B,85.72,85.40,84.97




disambig_acc


modification,Original,All Paraphrases,Baseline
target_model,,,
MPT-7B,41.07,40.79,40.82
MPT-7B-Inst,41.21,41.64,42.10
Falcon-7B,42.08,41.94,42.00
Falcon-7B-Inst,39.98,39.72,40.14
Llama-3-8B,57.20,56.88,55.87
Llama3-8B-Inst,30.36,29.59,30.08
Gemma3-1B,34.14,34.64,34.48
Gemma3-4B,68.97,69.60,69.10
Gemma3-12B,79.19,78.59,77.77




ambig_bias


modification,Original,All Paraphrases,Baseline
target_model,,,
MPT-7B,-0.17,0.52,1.13
MPT-7B-Inst,-0.17,0.51,1.52
Falcon-7B,-0.58,-1.21,-1.85
Falcon-7B-Inst,-0.75,-0.74,-0.10
Llama-3-8B,-2.95,-1.10,-1.36
Llama3-8B-Inst,1.60,-0.90,0.40
Gemma3-1B,-0.48,1.97,2.06
Gemma3-4B,-0.31,1.18,1.32
Gemma3-12B,0.21,0.59,0.80




disambig_bias


modification,Original,All Paraphrases,Baseline
target_model,,,
MPT-7B,2.45,1.82,2.48
MPT-7B-Inst,3.95,4.37,4.58
Falcon-7B,3.12,2.16,1.56
Falcon-7B-Inst,5.20,4.01,5.09
Llama-3-8B,-8.08,-6.30,-6.25
Llama3-8B-Inst,1.74,0.72,1.51
Gemma3-1B,-0.47,1.73,1.63
Gemma3-4B,-5.38,-4.45,-3.16
Gemma3-12B,-1.20,-2.04,-0.17


In [69]:
#Relative differences or differences to original, for baseline and AUGMENT settings
grouped = result_df.copy()

# Collapse modification values
grouped["modification"] = grouped["modification"].apply(
    lambda x: x if x in ["Original", "Baseline"] else "All Paraphrases"
)

# Group by target_model + modification
grouped = (
    grouped
    .groupby(["target_model", "modification"], as_index=False)
    .agg({metric: "mean" for metric in metrics_dataset.keys()})
)

for metric in metrics_dataset:
    # Pivot to wide format (one metric at a time)
    print('\n')
    print(metric)

    table = grouped.pivot(index="target_model", columns="modification", values=metric)

    # Reorder columns: Original → All Paraphrases → Random
    table = table[["Original", "All Paraphrases", "Baseline"]]
    
    if 'acc' in metric:
        # Compute relative difference to Original
        rel_table = ((table.sub(table["Original"], axis=0))
                     .div(table["Original"], axis=0) * 100)
    else:
        # Compute difference to Original
        rel_table = (table.sub(table["Original"], axis=0)) 

    # Reorder again
    rel_table = rel_table[["All Paraphrases", "Baseline"]]
    rel_table = rel_table.reindex(MODEL_ORDER)

    # Round to 2 decimals
    display(rel_table.round(2))



overall_acc


modification,All Paraphrases,Baseline
target_model,,
MPT-7B,-0.35,-0.07
MPT-7B-Inst,1.17,1.76
Falcon-7B,0.31,-0.19
Falcon-7B-Inst,-0.08,0.04
Llama-3-8B,-0.12,-1.26
Llama3-8B-Inst,0.58,1.02
Gemma3-1B,-0.20,-0.18
Gemma3-4B,-0.78,-0.96
Gemma3-12B,-0.73,-1.53




ambig_acc


modification,All Paraphrases,Baseline
target_model,,
MPT-7B,-0.04,2.10
MPT-7B-Inst,0.88,2.49
Falcon-7B,2.55,1.06
Falcon-7B-Inst,0.92,0.11
Llama-3-8B,1.35,3.29
Llama3-8B-Inst,2.86,2.49
Gemma3-1B,-1.78,-1.48
Gemma3-4B,-2.44,-2.12
Gemma3-12B,-0.29,-1.24




disambig_acc


modification,All Paraphrases,Baseline
target_model,,
MPT-7B,-0.53,-1.30
MPT-7B-Inst,1.32,1.37
Falcon-7B,-0.51,-0.65
Falcon-7B-Inst,-0.57,0.01
Llama-3-8B,-0.76,-3.22
Llama3-8B-Inst,-2.19,-0.76
Gemma3-1B,1.18,0.96
Gemma3-4B,0.33,-0.19
Gemma3-12B,-1.20,-1.85




ambig_bias


modification,All Paraphrases,Baseline
target_model,,
MPT-7B,0.01,0.01
MPT-7B-Inst,0.01,0.02
Falcon-7B,-0.01,-0.01
Falcon-7B-Inst,0.00,0.01
Llama-3-8B,0.02,0.02
Llama3-8B-Inst,-0.02,-0.01
Gemma3-1B,0.03,0.03
Gemma3-4B,0.02,0.02
Gemma3-12B,0.00,0.01




disambig_bias


modification,All Paraphrases,Baseline
target_model,,
MPT-7B,-0.01,0.00
MPT-7B-Inst,0.01,0.01
Falcon-7B,-0.02,-0.02
Falcon-7B-Inst,-0.01,-0.00
Llama-3-8B,0.02,0.02
Llama3-8B-Inst,-0.00,-0.00
Gemma3-1B,0.02,0.02
Gemma3-4B,0.01,0.02
Gemma3-12B,-0.00,0.01


In [ ]:
#Heatmaps of Relative differences or differences to original, per paraphrase type and across target models
for metric, title in metrics_dataset.items():
    # Compute means per modification and target model
    agg_df = result_df.groupby(["modification", "target_model"], as_index=False).mean(numeric_only=True)

    # Pivot full table so we can calculate relative differences
    pivot_full = agg_df.pivot(index="modification", columns="target_model", values=metric)

    # Make sure Original exists
    if "Original" not in pivot_full.index:
        raise ValueError("No 'Original' modification found in result_df")
        
    if 'acc' in metric:
        # Compute relative delta: (modification - Original) / Original
        diffs = ((pivot_full - pivot_full.loc["Original"]) / pivot_full.loc["Original"]) * 100
    else:
        # Compute difference to Original
        diffs = (pivot_full - pivot_full.loc["Original"]) 
    
    diffs = diffs.drop(index="Original")
    diffs = diffs.reindex(MODIF_ORDER)
    diffs=diffs[MODEL_ORDER]
    abs_max = diffs.abs().to_numpy().max()
    vmin, vmax = -abs_max, abs_max

    # Plot heatmap of relative deltas
    plt.figure(figsize=(10, 7))

    # Build cmap without alpha
    cmap = plt.get_cmap("vlag_r")
    if hasattr(cmap, "colors"):
        cmap = ListedColormap([c[:3] for c in cmap.colors], name="vlag_r_noalpha")

    # Apply with explicit norm centered at 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    ax = sns.heatmap(
        diffs, 
        annot=True, 
        fmt=".2f", 
        cmap=cmap, norm=norm,
        annot_kws={"size": 14}, 
        cbar_kws={"shrink": 0.8}
    )
    ax.collections[0].set_cmap(cmap)
    ax.hlines(1, *ax.get_xlim(), colors='white', linewidth=15)

    plt.ylabel("Paraphrase Type", fontsize=18)
    plt.xlabel("Target Model", fontsize=18)
    plt.xticks(fontsize=14, rotation=45, ha="right")
    plt.yticks(fontsize=14, rotation=0)

    colorbar = ax.collections[0].colorbar
    colorbar.ax.tick_params(labelsize=12)
    #colorbar.set_label(f"Δ {title} to Original (%)", fontsize=14)

    plt.tight_layout()
    #tikzplotlib.save(f"tex_files/{metric}_heatmap_{dataset}.tex")
    #plt.savefig(f"{metric}_heatmap_{dataset}.pdf", format="pdf", bbox_inches="tight")
    #plt.show()

In [ ]:
#Heatmaps of Relative differences or differences to original, per paraphrase type and per data subsets, for one target model
model="Gemma3-12B"# Focus on one target model
for metric, title in metrics_dataset.items():
    agg_df = (
        result_df[result_df.target_model == model]
        .groupby(["modification", "subset"], as_index=False)
        .mean(numeric_only=True)
    )

    # Pivot: rows = modification, cols = subset
    pivot_df = agg_df.pivot(index="modification", columns="subset", values=metric)

    # Compute absolute difference relative to Original
    if "Original" not in pivot_df.index:
        raise ValueError("No 'Original' modification found in result_df")
        
    if 'acc' in metric:
        # Compute relative delta: (modification - Original) / Original
        diffs = ((pivot_df - pivot_df.loc["Original"]) / pivot_df.loc["Original"]) * 100
    else:
        # Compute difference to Original
        diffs = (pivot_df - pivot_df.loc["Original"]) 

    # Drop Original row (so only relative differences show)
    diffs = diffs.drop(index="Original")
    diffs = diffs.reindex(MODIF_ORDER)
    diffs.columns = diffs.columns.str.replace("_", " ", regex=False).str.title()  
    abs_max = diffs.abs().to_numpy().max()
    vmin, vmax = -abs_max, abs_max

    # Build cmap without alpha
    cmap = plt.get_cmap("vlag_r")
    if hasattr(cmap, "colors"):
        cmap = ListedColormap([c[:3] for c in cmap.colors], name="vlag_r_noalpha")

    # Apply with explicit norm centered at 0
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # Plot
    plt.figure(figsize=(10, 7))
    ax=sns.heatmap(diffs, annot=True, fmt=".2f", cmap=cmap, norm=norm,
        annot_kws={"size": 14}, 
        cbar_kws={"shrink": 0.8})
    ax.collections[0].set_cmap(cmap)
    ax.hlines(1, *ax.get_xlim(), colors='white', linewidth=15)

    plt.ylabel("Paraphrase Type", fontsize=18)
    plt.xlabel("Dataset Subset", fontsize=18)
    plt.xticks(fontsize=14, rotation=45, ha="right")
    plt.yticks(fontsize=14, rotation=0)
    #plt.title(f"Δ {title} vs. Original ({model})")
    plt.tight_layout()
    #tikzplotlib.save(f"tex_files/{metric}_heatmap_{dataset}_subset.tex")
    #plt.show()